In [1]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# torch / torchvision 은 Colab 기본 버전을 그대로 씁니다. (건드리면 깨짐)
PACKAGES = [
    "transformers==4.55.2",
    "peft==0.17.0",
    "trl==0.21.0",
    "bitsandbytes==0.48.0",
    "accelerate==1.10.0",
    "datasets==4.0.0",
    "huggingface-hub<1.0",
]

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    print("설치 완료")
else:
    print("로컬 환경이라 설치를 건너뜁니다.")

설치 완료


In [2]:
import os

# GPU 메모리 단편화를 줄입니다. 리눅스 전용이며, torch가 CUDA를 잡기 전에 설정해야 합니다.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import transformers
import trl
from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("torch", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers", transformers.__version__, "| trl", trl.__version__)
if transformers.__version__ != "4.55.2" or trl.__version__ != "0.21.0":
    raise RuntimeError("라이브러리 버전이 다릅니다. 메뉴 '런타임 > 세션 다시 시작'을 누른 뒤 이 셀부터 다시 실행하세요.")

# torch의 CUDA 버전과 맞는 파일이 없으면 여기서 바로 오류가 납니다.
import bitsandbytes
print("bitsandbytes", bitsandbytes.__version__)

torch 2.11.0+cu130 | CUDA 13.0
transformers 4.55.2 | trl 0.21.0
bitsandbytes 0.48.0


In [3]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "qwen-koalpaca"

# 원래 4GB GPU에 맞춘 길이입니다. 이보다 긴 샘플은 자르지 않고 아예 뺍니다.
MAX_LENGTH = 256

TRAIN_SIZE = 500
VAL_SIZE = 100

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab 메뉴 '런타임 > 런타임 유형 변경'에서 T4 GPU를 고르세요.")
print(torch.cuda.get_device_name(0))

Tesla T4


In [4]:
raw = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
draw

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-21df739eb88d71(…):   0%|          | 0.00/12.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21155 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'output', 'url'],
    num_rows: 21155
})

In [6]:
import re

URL = re.compile(r"https?://")
SENTENCE_END = re.compile(r"[.!?…)\]\"'다요죠음됨함]\s*$")

def is_clean(item):
    answer = item["output"].strip()
    return not URL.search(answer) and bool(SENTENCE_END.search(answer))

clean_rows = raw.filter(is_clean)
len(raw), len(clean_rows)

Filter:   0%|          | 0/21155 [00:00<?, ? examples/s]

(21155, 20539)

In [7]:
def messages_for(item):
    return {
        "prompt": [{"role": "user", "content": item["instruction"]}],
        "completion": [{"role": "assistant", "content": item["output"]}],
    }

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def token_length(item):
    # 템플릿을 문자열로 만든 뒤 토큰으로 나눕니다. transformers 버전에 따라 결과가 달라지지 않습니다.
    m = messages_for(item)
    text = tokenizer.apply_chat_template(m["prompt"] + m["completion"], tokenize=False)
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
selected = []
seen_questions = set()
for item in clean_rows.shuffle(seed=42):
    question = item["instruction"].strip()
    if question in seen_questions:
        continue
    seen_questions.add(question)
    if token_length(item) <= MAX_LENGTH:
        selected.append(item)
        if len(selected) == TRAIN_SIZE + VAL_SIZE:
            break

train_set = selected[:TRAIN_SIZE]
val_set = selected[TRAIN_SIZE:]
len(train_set), len(val_set)

(500, 100)

In [10]:
messages_for(val_set[0])

{'prompt': [{'role': 'user', 'content': '컵라면을 최초로 만든 회사는 어디인가요?'}],
 'completion': [{'role': 'assistant',
   'content': "일본의 닛신식품에서 안도 모모후쿠(安藤百福) 회장이 인스턴트 라면을 발명하였습니다. 이후 1971년에는 끓는 물을 부어 잠시 후 바로 먹을 수 있는 '컵라면'을 개발하여 또 하나의 혁명을 일으켰습니다. 인기와 동시에 해외 진출이 이루어지며 세계적으로 인스턴트 라면의 선두주자로 자리매김하게 됩니다."}]}

In [11]:
train_ds = Dataset.from_list([messages_for(item) for item in train_set])
val_ds = Dataset.from_list([messages_for(item) for item in val_set])
train_ds

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 500
})

In [12]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [14]:
print("VRAM: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))

VRAM: 1.50 GB


In [15]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [16]:
train_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    completion_only_loss=True,      # 답변 토큰에만 손실을 겁니다
    num_train_epochs=1,
    per_device_train_batch_size=1,  # 2로 올리면 패딩 낭비로 오히려 느려집니다
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,    # 끄면 메모리가 넘쳐 2배 느려집니다
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,                      # Turing 세대라 bf16은 못 씁니다
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="epoch",          # 에폭 끝에 검증 손실을 계산합니다
    per_device_eval_batch_size=1,
    save_strategy="epoch",
    report_to=[],
    seed=42,
)

In [17]:
trainer = SFTTrainer(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.964600,1.996863


TrainOutput(global_step=63, training_loss=1.9839734198555115, metrics={'train_runtime': 316.7661, 'train_samples_per_second': 1.578, 'train_steps_per_second': 0.199, 'total_flos': 841440616611840.0, 'train_loss': 1.9839734198555115})

In [19]:
print("peak VRAM: %.2f GB" % (torch.cuda.max_memory_reserved() / 1024**3))

peak VRAM: 2.58 GB


In [20]:
trainer.save_model(OUTPUT_DIR)

In [22]:
import shutil

# False면 내 PC로 내려받고, True면 Google Drive에 저장합니다.
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/fine-tuning"

if IN_COLAB:
    zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    if SAVE_TO_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print("Drive에 저장:", shutil.copy(zip_path, DRIVE_DIR))
    else:
        from google.colab import files

        files.download(f"{OUTPUT_DIR}.zip")
else:
    print("로컬 환경이라 저장을 건너뜁니다. 결과는", OUTPUT_DIR, "폴더에 있습니다.")

Mounted at /content/drive
Drive에 저장: /content/drive/MyDrive/fine-tuning/qwen-koalpaca.zip


In [23]:
if "trainer" in globals():
    # 방금 학습한 모델을 그대로 씁니다
    tuned_model = trainer.model
    tuned_model.gradient_checkpointing_disable()
else:
    # 커널을 재시작했으면 저장된 어댑터를 불러옵니다
    from peft import PeftModel
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )
    tuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

tuned_model.eval()                        # 드롭아웃을 끕니다
tuned_model.config.use_cache = True       # KV 캐시를 켜야 생성이 빠릅니다
tokenizer.truncation_side = "left"        # 질문이 길면 뒤가 아니라 앞을 자릅니다

In [24]:
def answer_for(question, max_new_tokens=200):
    prompt = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        output = tuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [25]:
def compare(question):
    tuned = answer_for(question)
    with tuned_model.disable_adapter():
        base = answer_for(question)
    print("■ 질문\n" + question)
    print("\n■ 원래 Qwen\n" + base)
    print("\n■ 파인튜닝한 Qwen\n" + tuned)

In [26]:
compare(val_set[0]["instruction"])
print("\n■ 데이터의 답\n" + val_set[0]["output"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


■ 질문
컵라면을 최초로 만든 회사는 어디인가요?

■ 원래 Qwen
컵라면을 최초로 만든 회사 중 하나는 일본의 회사 "カプセルラーメン" (Kapshuuraam)입니다. 이 회사는 1978년에 창립되어 현재까지 계속해서 커피와 라면을 함께 판매하고 있습니다.

그러나, 실제 최초로 컵라면을 개발한 회사로서는 여러 가지 가능성이 있습니다:

1. 일본의 대형 음식점이나 식당에서 사용된 '카페라멘' 또는 '카페라멘'이라는 이름으로 알려진 제품이 있었을 수 있습니다.
2. 일본의 대형 음식점이나 식당에서 사용된 '카페라멘' 또는 '카페라멘'이라는 이름으로 알려진 제품이 있었을 수 있습니다.
3. 일본의 대형 음식점이나 식당에서 사용된 '카페라멘

■ 파인튜닝한 Qwen
컵라면은 1930년대에 일본에서 만들어졌습니다. 이때는 페트병이나 컵으로 냄비를 사용했지만, 이후에는 일반적인 커프컵으로 바꾸어 사용되었습니다. 현재도 가장 많이 사용되는 커프컵이죠.

■ 데이터의 답
일본의 닛신식품에서 안도 모모후쿠(安藤百福) 회장이 인스턴트 라면을 발명하였습니다. 이후 1971년에는 끓는 물을 부어 잠시 후 바로 먹을 수 있는 '컵라면'을 개발하여 또 하나의 혁명을 일으켰습니다. 인기와 동시에 해외 진출이 이루어지며 세계적으로 인스턴트 라면의 선두주자로 자리매김하게 됩니다.


In [27]:
compare(val_set[1]["instruction"])
print("\n■ 데이터의 답\n" + val_set[1]["output"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


■ 질문
국내에서 우측 운전석을 가진 차량으로 운전해도 법적으로 문제가 없는가요?

■ 원래 Qwen
물론이죠. 국내에서는 모든 차량의 운전석은 동일하게 사용할 수 있습니다. 따라서 우측 운전석을 가진 차량으로 운전해도 법적으로 문제가 없습니다. 다만, 주행 시 안전을 위해 좌측 운전석을 이용하는 것이 좋습니다. 이는 다른 운전자의 눈에 보이는 위치를 유지하고, 사고 시 안전벨트와 헬멧 등을 적절히 착용하는 데 도움이 됩니다. 또한, 교통법규나 규정에 따라 특정 지역이나 시간대에선 일부 차량이 제한될 수도 있으니 참고하시기 바랍니다.

■ 파인튜닝한 Qwen
물론이죠. 국내에서는 모든 차량의 운전석은 좌측에 있어야 합니다. 하지만 우측 운전석을 사용하는 경우는 가능합니다. 다만, 이 경우에는 주행거리와 주행시간이 길어질 수 있으므로, 적절한 조치를 취해야 합니다. 예를 들어, 우측 운전석을 사용할 때는 주행거리와 주행시간이 긴다면, 우측 운전석을 사용하기 전에 주행거리와 주행시간을 계산하여 적절한 조치를 취해야 합니다. 또한, 우측 운전석을 사용할 때는 주행거리와 주행시간이 긴다면, 우측 운전석을 사용하기 전에 주행거리와 주행시간을 계산하여 적절한 조치를 취해야 합니다.

■ 데이터의 답
법적으로는 전혀 문제 되지 않습니다. 국내에서 돌아다니는 일본 차량들 중에서도 우측 운전석 차량을 타는 사람이 있으며, 그런 차들로 운전해도 무방합니다. 단, 우측 운전석 차량을 타는 것은 적응이 필요하고, 교통안전에 이슈가 있을 수 있으므로 주의가 필요합니다. 또한, 통행료를 낼 때 같은 차선에서 운전하는 차량과의 거리를 유지해야 합니다.


In [28]:
REFUSAL = re.compile(r"인공지능|AI 모델|AI로서|언어 ?모델|죄송|알려드리기 어렵")
COUNT_SIZE = 10

tuned_count = base_count = 0
for item in val_set[:COUNT_SIZE]:
    tuned_count += bool(REFUSAL.search(answer_for(item["instruction"])))
    with tuned_model.disable_adapter():
        base_count += bool(REFUSAL.search(answer_for(item["instruction"])))

print(f"거절 표현이 들어간 답: 원래 Qwen {base_count}/{COUNT_SIZE}, 파인튜닝한 Qwen {tuned_count}/{COUNT_SIZE}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

거절 표현이 들어간 답: 원래 Qwen 1/10, 파인튜닝한 Qwen 0/10


In [29]:
compare("겨울철 실내 습도를 적당히 유지하는 방법은 무엇인가요?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


■ 질문
겨울철 실내 습도를 적당히 유지하는 방법은 무엇인가요?

■ 원래 Qwen
겨울철 실내 습도를 적당히 유지하는 것은 건강과 안전을 위해 중요합니다. 다음과 같은 방법들을 통해 습도를 조절할 수 있습니다:

1. 창문을 열어주는 방식: 창문을 열면 자연스럽게 습기를 빼낼 수 있습니다.

2. 습도 제어기 사용: 가정용 습도제어기가 있다면 그에 따라 습도를 조절할 수 있습니다.

3. 물을 덜 때마다 청소: 습도가 증가하면 물이 더 많이 발생하므로 이를 줄이는 것이 좋습니다.

4. 창구나 화목기 사용: 이들은 습기를 빨리 떨어뜨릴 수 있습니다.

5. 창고나 보관함 등에서 물을 덜 때마다 청소: 습도가 증가하면

■ 파인튜닝한 Qwen
겨울철에는 실내 습도를 적당히 유지하는 것이 중요합니다. 이는 건강을 보호하고 집안의 환경을 개선하는데 도움이 됩니다. 아래와 같이 실내 습도를 조절할 수 있는 방법들을 소개해 드립니다.

1. 창문을 열어주는 방식: 창문을 열어주면 자연스럽게 실내 습도를 조절할 수 있습니다. 하지만 창문을 열면 떨어지는 날씨에 따라 창문이 닫히지 않도록 주의해야 합니다.

2. 스크린이나 필터를 사용하는 방식: 스크린이나 필터를 사용하면 실내 습기를 제거하거나, 바람을 통과시키면서 습기를 흡수하여 실내 습도를 줄일 수 있습니다.

3. �


In [30]:
compare("클로드 코드가 더 똑똑할까? 아니면 제미나이가 더 똑똑할까?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


■ 질문
클로드 코드가 더 똑똑할까? 아니면 제미나이가 더 똑똑할까?

■ 원래 Qwen
이 질문은 매우 복잡하고 개인적인 의견을 표현하는 것처럼 보입니다. 저는 AI로서, 인간의 감정이나 지적 능력에 대해 이해하거나 판단하지 못합니다.

그러나, 클로드 코드와 제미나이는 모두 인공지능으로, 각각의 특징과 기술적 성능을 가지고 있습니다. 따라서 어떤 한쪽이 더 똑똑하다는 것은 불가능한 것이며, 두 가지 모두가 서로 다른 방식으로 정보를 처리하고 학습하며 발전할 수 있다는 점도 중요합니다.

결국, 어떤 AI가 더 똑똑할지는 개인의 가치관과 선호도에 따라 달라질 수 있습니다. 중요한 것은 AI가 사용자에게 도움이 되고, 그들이 원하는 정보를 쉽게 얻을 수 있도록 돕는다는 것입니다.

■ 파인튜닝한 Qwen
두 개의 AI 모델은 각각 다른 특징과 기능을 가지고 있습니다. 클로드 코드는 자연어 처리와 인공지능 기반의 언어학적 기능을 제공하며, 제미나이는 음성인식 및 텍스트 인코딩 기능을 제공합니다. 따라서 두 모델은 서로 다른 방식으로 사용자에게 도움을 줄 수 있습니다. 따라서 어떤 모델이 더 똑똑하다고 말하기는 어렵습니다. 중요한 것은 사용자가 필요로 하는 서비스를 제공하는 것이며, 이에 따라 선택해야 할 모델이 달라질 수 있다는 것입니다.


In [31]:
compare("수업시간에 자도 될까?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


■ 질문
수업시간에 자도 될까?

■ 원래 Qwen
물론이죠! 수업 시간에는 항상 학습을 위해 최선을 다하겠습니다. 질문이나 도움이 필요하면 언제든지 말씀してください。

■ 파인튜닝한 Qwen
물론이죠! 수업 시간에는 학습을 위해 필요한 정보를 공부하고 배울 수 있습니다. 하지만, 학생들이 수업 시간에 자는 것은 부적절한 행동입니다. 교사와 학생 모두에게 불편함을 주고, 학교의 정책과 규칙에도 위반됩니다. 따라서, 수업 시간에는 학생들은 수업 내용을认真听和认真做笔记，以便在课后复习时能够更好地理解所学内容。
